# Mô phỏng Sentinel-1 GRD DN và grayscale

Notebook này mô phỏng band **Amplitude DN** của Sentinel-1 Level-1 GRD, sau đó:

1. Loại no-data khỏi tập thống kê;
2. Tính minimum, maximum, mean, standard deviation, median và histogram;
3. Chọn display range gần với hai cách của SNAP;
4. Chuyển DN sang chỉ số grayscale `0..255`;
5. Hiển thị kết quả theo palette `black → white`.

> Đây là mô phỏng logic, không phải bản port từng dòng mã Java của SNAP. SNAP tìm range trên các histogram bin; notebook dùng quantile trực tiếp nên kết quả có thể lệch rất nhỏ tại biên bin.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["image.cmap"] = "gray"

## 1. DN trong Sentinel-1 GRD có ý nghĩa gì?

Theo [Sentinel-1 Product Specification](https://sentiwiki.copernicus.eu/__attachments/1673968/S1-RS-MDA-52-7441%20-%20Sentinel-1%20Product%20Specification%202025%20-%203.16.1.pdf), measurement TIFF của Level-1 GRD lưu mỗi pixel dưới dạng **một magnitude sample `uint16`**:

- Miền lưu trữ có thể biểu diễn: `0..65535`;
- SNAP đọc band này thành `Amplitude_<polarisation>` với unit `amplitude`;
- SNAP dùng `DN = 0` làm no-data cho Sentinel-1 Level-1;
- Band `Intensity_<polarisation>` trong SNAP là virtual band: $Intensity = DN^2$;
- Với SLC, intensity được tính từ hai thành phần phức: $Intensity = I^2 + Q^2$; notebook này chỉ mô phỏng **GRD**.

DN là số đã được IPF scale để lưu vào 16 bit, **không phải trực tiếp là $\sigma^0$ và không có đơn vị vật lý cố định**. Muốn hiệu chuẩn pixel $i$, phải dùng LUT `sigmaNought`, `betaNought` hoặc `gamma` trong chính SAFE product:

$$calibratedValue_i = \frac{DN_i^2}{A_i^2}$$

Trong đó $A_i$ được nội suy từ calibration LUT và thay đổi theo range/product. Vì vậy **không tồn tại một bảng DN → loại bề mặt đúng cho mọi ảnh**. Polarisation, góc tới, độ ẩm, độ nhám, hình học và LUT của product đều làm DN thay đổi.

Khoảng dùng trong mô phỏng dưới đây chỉ để dễ hình dung, không phải ngưỡng phân loại Sentinel-1:

| Đối tượng mô phỏng | Amplitude DN điển hình trong notebook |
|---|---:|
| Nước phẳng | khoảng `100–800` |
| Ruộng/đất/thảm thực vật | khoảng `1,000–4,000` |
| Đô thị/đường | khoảng `4,000–12,000` |
| Điểm phản xạ rất mạnh | khoảng `15,000–60,000` |
| No-data | `0` |

GRD là magnitude/amplitude, nên speckle Gamma được tạo trong miền intensity rồi lấy căn bậc hai để tác động lên DN amplitude. `looks = 4` gần với ENL trung bình của IW GRD HR và cho speckle nhẹ hơn single-look.

In [ ]:
height, width = 360, 480
looks = 4
y, x = np.mgrid[:height, :width]
yn, xn = y / (height - 1), x / (width - 1)

# Texture không gian tương quan nhẹ, gần với biến thiên địa hình hơn nhiễu độc lập.
texture = rng.normal(size=(height, width))
for _ in range(10):
    texture = (texture + np.roll(texture, 1, 0) + np.roll(texture, -1, 0) + np.roll(texture, 1, 1) + np.roll(texture, -1, 1)) / 5
texture /= texture.std()

# Amplitude DN trung bình của nền rừng/đất; giảm nhẹ theo hướng range.
range_trend = 1.0 - 0.25 * xn
base_dn = 2800 * np.exp(0.12 * texture) * range_trend

# Các thửa ruộng có mức tán xạ khác nhau.
parcels = [
    (25, 105, 25, 135, 1300), (25, 105, 140, 225, 2400),
    (112, 190, 20, 105, 1700), (112, 190, 110, 225, 3600),
    (205, 320, 25, 120, 2100), (205, 320, 125, 230, 1100),
]
for y0, y1, x0, x1, level_dn in parcels:
    base_dn[y0:y1, x0:x1] = level_dn * np.exp(0.08 * texture[y0:y1, x0:x1]) * range_trend[y0:y1, x0:x1]

# Nước phẳng thường rất tối trên SAR: một sông uốn khúc và một hồ.
river_center = 0.50 + 0.075 * np.sin(2 * np.pi * (1.3 * xn + 0.05))
river = np.abs(yn - river_center) < 0.022 + 0.006 * np.cos(4 * np.pi * xn)
lake = ((xn - 0.78) / 0.13) ** 2 + ((yn - 0.22) / 0.10) ** 2 < 1
water = river | lake
base_dn[water] = 350 * np.exp(0.10 * texture[water])

# Đô thị sáng, có cấu trúc ô phố; đường/đê là dải phản xạ tương đối mạnh.
urban = (xn > 0.62) & (xn < 0.93) & (yn > 0.58) & (yn < 0.91)
base_dn[urban] = 5200 * np.exp(0.15 * texture[urban])
building_lines = urban & (((x % 24) < 3) | ((y % 20) < 3))
base_dn[building_lines] = 9000 * np.exp(0.08 * texture[building_lines])
road = np.abs(yn - (0.86 - 0.52 * xn)) < 0.006
base_dn[road & ~water] = 6500

# Speckle Gamma tác động lên intensity; GRD DN lưu magnitude nên dùng sqrt(speckle).
speckle = rng.gamma(shape=looks, scale=1.0 / looks, size=base_dn.shape)
sar_dn = np.clip(np.rint(base_dn * np.sqrt(speckle)), 1, 65535).astype(np.uint16)

# Corner reflectors/kim loại tạo các điểm rất sáng trong vùng đô thị.
urban_y, urban_x = np.where(urban)
bright = rng.choice(urban_y.size, size=35, replace=False)
sar_dn[urban_y[bright], urban_x[bright]] = rng.integers(15000, 45001, bright.size, dtype=np.uint16)

# No-data ở biên/góc khuyết; vài outlier giúp minh họa robust stretch.
sar_dn[:6, :] = sar_dn[-6:, :] = 0
sar_dn[:, :6] = sar_dn[:, -6:] = 0
sar_dn[(x < 45) & (y < 85 - 1.5 * x)] = 0
outlier_y = rng.integers(10, height - 10, 8)
outlier_x = rng.integers(10, width - 10, 8)
sar_dn[outlier_y[:4], outlier_x[:4]] = np.array([60000, 58000, 56000, 54000], dtype=np.uint16)
sar_dn[outlier_y[4:], outlier_x[4:]] = 1

fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
axes[0].imshow(base_dn, vmin=0, vmax=10000, cmap="gray")
axes[0].set_title("Amplitude DN trung bình trước speckle")
im = axes[1].imshow(np.ma.masked_equal(sar_dn, 0), vmin=0, vmax=10000, cmap="gray")
axes[1].set_title("Sentinel-1 GRD DN mô phỏng (uint16)")
fig.colorbar(im, ax=axes[1], label="Amplitude DN")
for ax in axes:
    ax.axis("off")
plt.show()

## 2. Tính statistics trên valid pixels

No-data phải bị loại trước. Nếu đưa no-data vào histogram, display range có thể bị kéo lệch và ảnh mất tương phản.

Hai range được tính:

- **SNAP core-like default:** bỏ 1% đuôi trái và 4% đuôi phải, giữ khoảng 95% dữ liệu;
- **Colour Manipulation `From Data` mặc định:** giữ 92% ở giữa, tức bỏ 4% mỗi phía.

In [ ]:
valid_mask = sar_dn != 0
valid = sar_dn[valid_mask]

statistics = {
    "valid_count": valid.size,
    "no_data_count": sar_dn.size - valid.size,
    "minimum_dn": valid.min(),
    "maximum_dn": valid.max(),
    "mean_dn": valid.mean(),
    "std_dn": valid.std(),
    "median_dn": np.median(valid),
}

for name, value in statistics.items():
    print(f"{name:>16}: {value:,.4f}" if isinstance(value, (float, np.floating)) else f"{name:>16}: {value:,}")

snap_min, snap_max = np.quantile(valid, [0.01, 0.96])
from_data_min, from_data_max = np.quantile(valid, [0.04, 0.96])

print(f"\nAbsolute range       : [{valid.min():.0f}, {valid.max():.0f}] DN")
print(f"SNAP core-like range : [{snap_min:.1f}, {snap_max:.1f}] DN")
print(f"From Data 92% range  : [{from_data_min:.1f}, {from_data_max:.1f}] DN")

In [ ]:
counts, edges = np.histogram(valid, bins=256)

fig, ax = plt.subplots(figsize=(12, 4), constrained_layout=True)
ax.hist(valid, bins=256, color="0.55")
ax.axvline(valid.min(), color="tab:red", linestyle=":", label="Absolute min/max")
ax.axvline(valid.max(), color="tab:red", linestyle=":")
ax.axvline(snap_min, color="tab:blue", linestyle="--", label="SNAP core-like 1% / 4%")
ax.axvline(snap_max, color="tab:blue", linestyle="--")
ax.axvline(from_data_min, color="tab:green", label="From Data 92%")
ax.axvline(from_data_max, color="tab:green")
ax.set(title="Histogram valid Sentinel-1 GRD samples", xlabel="Amplitude DN", ylabel="Pixel count")
ax.legend()
plt.show()

## 3. Chuyển display range sang grayscale

DN được ánh xạ tuyến tính vào palette grayscale:

$$t = \mathrm{clip}\left(\frac{g-Min}{Max-Min}, 0, 1\right)$$

$$gray = \mathrm{round}(255t)$$

- `g ≤ Min` → `0` → black;
- `g ≥ Max` → `255` → white;
- Giá trị ở giữa được nội suy tuyến tính.

In [ ]:
def to_grayscale(values, display_min, display_max, no_data=None):
    """Map valid DN values to uint8 grayscale and return the validity mask."""
    if not display_max > display_min:
        raise ValueError("display_max must be greater than display_min")
    mask = np.isfinite(values)
    if no_data is not None:
        mask &= values != no_data
    gray = np.zeros(values.shape, dtype=np.uint8)
    normalized = np.clip((values[mask] - display_min) / (display_max - display_min), 0.0, 1.0)
    gray[mask] = np.rint(255.0 * normalized).astype(np.uint8)
    return gray, mask

gray_absolute, _ = to_grayscale(sar_dn, valid.min(), valid.max(), no_data=0)
gray_snap, gray_valid = to_grayscale(sar_dn, snap_min, snap_max, no_data=0)
gray_from_data, _ = to_grayscale(sar_dn, from_data_min, from_data_max, no_data=0)

# Self-check nhỏ: clipping và kiểu dữ liệu phải đúng.
probe, probe_mask = to_grayscale(np.array([-1.0, 0.0, 1.0, 2.0, 3.0, np.nan]), 0.0, 2.0)
assert probe.tolist() == [0, 0, 128, 255, 255, 0]
assert probe.dtype == np.uint8 and probe_mask.tolist() == [True, True, True, True, True, False]
print("Self-check passed:", probe.tolist())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)
images = [
    (gray_absolute, "Absolute min/max\n(outlier làm giảm tương phản)"),
    (gray_snap, "SNAP core-like\n1% left / 4% right"),
    (gray_from_data, "From Data\ncentral 92%"),
]

for ax, (image, title) in zip(axes, images):
    shown = np.ma.array(image, mask=~valid_mask)
    cmap = plt.colormaps["gray"].copy()
    cmap.set_bad("yellow")  # Chỉ để nhìn rõ no-data trong mô phỏng.
    ax.imshow(shown, cmap=cmap, vmin=0, vmax=255)
    ax.set_title(title)
    ax.axis("off")

plt.show()

Màu vàng trong hình chỉ đánh dấu no-data để dễ quan sát. Mảng kết quả thật vẫn là grayscale `uint8`; no-data được giữ riêng bằng `valid_mask`, tương tự cách SNAP dùng valid-mask/no-data color sau bước tạo ảnh màu.

In [ ]:
samples = np.linspace(max(0, snap_min - 500), snap_max + 500, 500)
indices, _ = to_grayscale(samples, snap_min, snap_max)

fig, ax = plt.subplots(figsize=(10, 4), constrained_layout=True)
ax.plot(samples, indices, color="black")
ax.axvline(snap_min, color="tab:blue", linestyle="--", label="Display Min")
ax.axvline(snap_max, color="tab:red", linestyle="--", label="Display Max")
ax.set(xlabel="Amplitude DN", ylabel="Grayscale index", ylim=(-5, 260), title="Ánh xạ DN → grayscale 0..255")
ax.legend()
plt.show()

## 4. Tương tác trực tiếp với Sentinel-1 GRD DN

Cell này ánh xạ trực tiếp **Amplitude DN `uint16`** từ measurement TIFF giả lập. Với full range `1..60000`, một pixel nền `DN ≈ 2000` chỉ nhận gray khoảng `8/255`, nên gần như black. Thanh **From Data** loại các đuôi histogram để khôi phục tương phản; click ảnh để xem DN và grayscale. Các điều khiển không sửa `sar_dn`.

In [ ]:
%matplotlib widget
import ipywidgets as widgets
from IPython.display import display

interactive_data = sar_dn.astype(float)
interactive_valid = interactive_data[valid_mask]
interactive_from_data_min, interactive_from_data_max = np.quantile(interactive_valid, [0.04, 0.96])

interactive_cmap = plt.colormaps["gray"].copy()
interactive_cmap.set_bad("magenta")
shown = np.ma.array(interactive_data, mask=~valid_mask)

fig, (ax_image, ax_hist) = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
image_artist = ax_image.imshow(shown, cmap=interactive_cmap, vmin=interactive_from_data_min, vmax=interactive_from_data_max)
ax_image.set_title("Click để đọc pixel")
ax_image.axis("off")
ax_hist.hist(interactive_valid, bins=256, color="0.65")
min_line = ax_hist.axvline(interactive_from_data_min, color="tab:blue", linewidth=2)
max_line = ax_hist.axvline(interactive_from_data_max, color="tab:red", linewidth=2)
ax_hist.set(title="Display range trên histogram", xlabel="Amplitude DN", ylabel="Pixel count")

range_slider = widgets.FloatRangeSlider(
    value=(interactive_from_data_min, interactive_from_data_max),
    min=float(interactive_valid.min()), max=float(interactive_valid.max()),
    step=1, description="Min / Max [DN]", continuous_update=True,
    readout_format=".0f", style={"description_width": "140px"},
    layout=widgets.Layout(width="95%"),
)
percentile_slider = widgets.FloatSlider(
    value=92, min=70, max=100, step=1, description="From Data [%]",
    continuous_update=False, style={"description_width": "140px"},
    layout=widgets.Layout(width="70%"),
)
full_range_button = widgets.Button(description="Full range")
snap_button = widgets.Button(description="SNAP core 1% / 4%")
range_info = widgets.HTML()
pixel_info = widgets.HTML("Click một pixel trên ảnh để xem phép ánh xạ.")

def update_range(change=None):
    """Apply the selected display range without changing source samples."""
    display_min, display_max = range_slider.value
    image_artist.set_clim(display_min, display_max)
    min_line.set_xdata([display_min, display_min])
    max_line.set_xdata([display_max, display_max])
    clipped = 100 * np.mean((interactive_valid <= display_min) | (interactive_valid >= display_max))
    range_info.value = f"DN range: <b>[{display_min:.0f}, {display_max:.0f}]</b> — clipped: {clipped:.2f}%"
    fig.canvas.draw_idle()

def use_central_percentile(change):
    """Set a symmetric range containing the requested central percentage."""
    tail = (100 - percentile_slider.value) / 200
    range_slider.value = tuple(np.quantile(interactive_valid, [tail, 1 - tail]))

def use_snap_default(_):
    """Restore the SNAP core-like asymmetric percentile range."""
    range_slider.value = (snap_min, snap_max)

def use_full_range(_):
    """Use the absolute minimum and maximum, including outliers."""
    range_slider.value = (float(interactive_valid.min()), float(interactive_valid.max()))

def inspect_pixel(event):
    """Report geophysical and grayscale values for the clicked valid pixel."""
    if event.inaxes is not ax_image or event.xdata is None or event.ydata is None:
        return
    px, py = int(round(event.xdata)), int(round(event.ydata))
    if not (0 <= px < width and 0 <= py < height) or not valid_mask[py, px]:
        pixel_info.value = f"Pixel (x={px}, y={py}) là no-data."
        return
    display_min, display_max = range_slider.value
    value_dn = interactive_data[py, px]
    gray_value = int(np.rint(255 * np.clip((value_dn - display_min) / (display_max - display_min), 0, 1)))
    pixel_info.value = f"Pixel <b>(x={px}, y={py})</b>: DN=<b>{value_dn:.0f}</b> → gray <b>{gray_value}</b> → RGB({gray_value}, {gray_value}, {gray_value})"

range_slider.observe(update_range, names="value")
percentile_slider.observe(use_central_percentile, names="value")
full_range_button.on_click(use_full_range)
snap_button.on_click(use_snap_default)
fig.canvas.mpl_connect("button_press_event", inspect_pixel)
update_range()
display(widgets.VBox([range_slider, widgets.HBox([percentile_slider, full_range_button, snap_button]), range_info, pixel_info]))
plt.show()

## 5. Kết luận

```text
Sentinel-1 GRD Amplitude DN (uint16)
    → bỏ no-data
    → statistics + histogram
    → chọn display Min/Max
    → clip và chuẩn hóa 0..1
    → lượng tử hóa 0..255
    → gray index i
    → RGB(i, i, i)
```

Thay đổi display range chỉ thay đổi độ tương phản hiển thị. Mảng `sar_dn` không bị sửa. DN chỉ là magnitude đã scale để lưu trữ; muốn suy ra backscatter vật lý phải dùng calibration LUT của chính product.